# Exercise 4: Projecting tags onto parallel Turkic sentences

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Endangered Languages and Ethics · version 0.1.0*

> Project tags onto aligned target sentences and compare with independent gold on those exact sentences. Report unaligned cases and keep the evaluation sentences out of training.

**Setup.** `pip install requests huggingface_hub fasttext-wheel transformers torch sentencepiece "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this, and Exercise 1 also calls `git`. Data come from commit- or tag-pinned GitHub files and are hash-checked: the UD 2.17 TueCL treebanks, the KazNERD and KyrgyzNER releases and their README files, and the TIL corpus README. The audit in Exercise 1 also reads the book's UD 2.17 inventory (`tables/tbl-ud217-inventory/data.csv`), the NLLB-200 tokeniser (`NLLB_PATH`) and GlotLID (`GLOTLID_PATH`), and queries live services, so its live columns are dated observations, not pinned results. Run the notebooks from the `exercises/` folder: the first code cell imports the chapter's `observation` and `hub_count` helpers from `../code/remaining/resources.py`. Downloads are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`).

The UD 2.17 TueCL treebanks contain the same sentences in Turkish, Azerbaijani, Uzbek and Kyrgyz, linked by `parallel_id`, each annotated by hand. Turkish gold UPOS tags are projected through two word aligners: a monotone string-similarity aligner using CTS for target words and native Turkish for source words, and a SimAlign-style mutual argmax over XLM-R layer-8 word vectors. Unaligned target words receive no tag.

In [ ]:
import collections, csv, datetime, difflib, hashlib, json, os, re, statistics, subprocess, sys, time
from pathlib import Path
import requests
sys.path.insert(0, str(Path("../code/remaining").resolve()))
from resources import observation, hub_count

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/"
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
UD = "https://raw.githubusercontent.com/UniversalDependencies/{}/r2.17/{}"
SHA256 = {
    KAZNERD + "README.md": "2d56859aa35eadafb8bc560b375d5128627e3bab3517bc74377c7818905e3614",
    KAZNERD + "KazNERD/IOB2_train.txt": "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KAZNERD + "KazNERD/IOB2_test.txt": "359a1f61f6a21280c6c47b4fde9a5adcd07411d62015aa6e32d940c51f163b7a",
    KYRGYZNER + "README.md": "c5beec0dd94156f5cdae4c5f7ffafeb23704f6fd1024ed4cb505cc8ef498a46c",
    TIL + "til_corpus/README.md": "da88f81cc7a23fb46425dcf236f7c89ad0f63663d2141d8fb7165ea8015308d8",
    UD.format("UD_Turkish-TueCL", "tr_tuecl-ud-test.conllu"): "2508f47e57b9b6a1d7241d0ab4a918d84c5cbb0455bbd961ac5231e31a1d311e",
    UD.format("UD_Azerbaijani-TueCL", "az_tuecl-ud-test.conllu"): "8b315f8b065f094e38482465706ba44a9caf1b082fc8bbc94908f6fdeee8c3b5",
    UD.format("UD_Uzbek-TueCL", "uz_tuecl-ud-test.conllu"): "23257711196df0bbfaf5b25397d7c6102fbc8800ed83b15eb77342929ee0e469",
    UD.format("UD_Kyrgyz-TueCL", "ky_tuecl-ud-test.conllu"): "08224be6b74346f582234f54527412dd6265c8d8b42e7d6ef463aade85bdcd9b",
}
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256(path.read_bytes()).hexdigest(); FETCHED[url] = h
    if url in SHA256 and h != SHA256[url]:
        raise ValueError(f"{url} differs from the pinned file")
    return path

def read_conll(path):
    """KazNERD-style column file -> [(words, tags)]."""
    sents, words, tags = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words: sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith("# "):
            c = line.split("\t") if "\t" in line else line.split()
            words.append(c[0]); tags.append(c[-1].strip())
    return sents

def show(obs):
    """One audit cell: the value, or the reason why there is none. Zero, not found, error and not checked stay distinct."""
    return {"observed": lambda o: str(o["value"]), "not_found": lambda o: "not found", "error": lambda o: "ERROR",
            "not_checked": lambda o: "not checked"}[obs["status"]](obs)
print("helpers loaded from code/remaining/resources.py:", observation.__name__, hub_count.__name__)

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
def read_ud(url):
    sents, cur, meta = {}, [], {}
    for line in fetch(url).read_text(encoding="utf-8").splitlines() + [""]:
        if line.startswith("# parallel_id"): meta["pid"] = line.split("=", 1)[1].strip()
        elif not line.strip():
            if cur: sents[meta["pid"]] = cur
            cur, meta = [], {}
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit(): cur.append((c[1], c[3]))
    return sents
TB = {"tr": ("Turkish", "tur"), "az": ("Azerbaijani", "aze"), "uz": ("Uzbek", "uzb"), "ky": ("Kyrgyz", "kir")}
data = {c: read_ud(UD.format(f"UD_{n}-TueCL", f"{c}_tuecl-ud-test.conllu")) for c, (n, _) in TB.items()}
_tr = {}
def norm(lang, w):
    """Lower-cased Common Turkic Alphabet form, so string similarity can cross scripts (Turkish is left as is)."""
    if lang == "tur": return w.lower()
    if lang not in _tr:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        _tr[lang] = Transliterator(lang, Script.CYRILLIC if lang == "kir" else Script.LATIN, Script.COMMON_TURKIC)
    return _tr[lang].transliterate(w).lower()
def string_align(src, tgt, s_lang, t_lang, threshold=0.5):
    """Monotone alignment maximising string similarity (difflib ratio) minus a position penalty; pairs below the threshold stay unaligned."""
    a = [norm(s_lang, w) for w in src]; b = [norm(t_lang, w) for w in tgt]
    S = [[difflib.SequenceMatcher(None, x, y).ratio() - abs(i / len(a) - j / len(b)) for j, y in enumerate(b)] for i, x in enumerate(a)]
    best = [[0.0] * (len(b) + 1) for _ in range(len(a) + 1)]; back = {}
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            opts = [(best[i - 1][j], "up"), (best[i][j - 1], "left")]
            if S[i - 1][j - 1] >= threshold: opts.append((best[i - 1][j - 1] + S[i - 1][j - 1], "diag"))
            best[i][j], back[i, j] = max(opts)
    i, j, links = len(a), len(b), {}
    while i and j:
        m = back[i, j]
        if m == "diag": links[j - 1] = i - 1; i, j = i - 1, j - 1
        elif m == "up": i -= 1
        else: j -= 1
    return links                                    # target index -> source index
src = "sentence-transformers/paraphrase-xlm-r-multilingual-v1"
local = Path(os.environ.get("MODELS_DIR", CACHE / "models")) / "models--sentence-transformers--paraphrase-xlm-r-multilingual-v1--c2e19453aae46f98a865abfdf173f35f2e119f5a"
tok = AutoTokenizer.from_pretrained(local if local.exists() else src, revision=None if local.exists() else "c2e19453aae46f98a865abfdf173f35f2e119f5a")
enc = AutoModel.from_pretrained(local if local.exists() else src, revision=None if local.exists() else "c2e19453aae46f98a865abfdf173f35f2e119f5a").eval()
def word_vectors(words, layer=8):
    b = tok(words, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=256)
    with torch.inference_mode():
        h = enc(**b, output_hidden_states=True).hidden_states[layer][0]
    ids = b.word_ids(); out = []
    for k in range(len(words)):
        rows = [i for i, w in enumerate(ids) if w == k]
        out.append(h[rows].mean(0) if rows else torch.zeros(h.shape[-1]))
    return torch.nn.functional.normalize(torch.stack(out), dim=-1)
def embed_align(src_words, tgt_words):
    """SimAlign-style mutual argmax on layer-8 word vectors: link only words that choose each other."""
    sim = word_vectors(tgt_words) @ word_vectors(src_words).T
    fwd, bwd = sim.argmax(1), sim.argmax(0)
    return {j: int(fwd[j]) for j in range(len(tgt_words)) if int(bwd[fwd[j]]) == j}
print(f"UD 2.17 TueCL test files: {', '.join(f'{c} {len(v)}' for c, v in data.items())} sentences; aligned through parallel_id")
for t in ("az", "uz", "ky"):
    pids = [p for p in data["tr"] if p in data[t]]
    gold_all = [u for p in pids for _, u in data[t][p]]
    majority = collections.Counter(gold_all).most_common(1)[0]
    print(f"\nTurkish -> {TB[t][0]}: {len(pids)} parallel sentences, {len(gold_all)} target words; "
          f"majority tag {majority[0]} = {majority[1] / len(gold_all):.1%}")
    for name in ("string+position", "XLM-R mutual argmax"):
        n_al = n_ok = 0; conf = collections.Counter(); unal = collections.Counter()
        for p in pids:
            s, tg = data["tr"][p], data[t][p]
            links = string_align([w for w, _ in s], [w for w, _ in tg], "tur", TB[t][1]) if name == "string+position" \
                    else embed_align([w for w, _ in s], [w for w, _ in tg])
            for j, (w, u) in enumerate(tg):
                if j in links:
                    n_al += 1; pu = s[links[j]][1]; n_ok += pu == u
                    if pu != u: conf[(pu, u)] += 1
                else:
                    unal[u] += 1
        print(f"  {name:<20} aligned {n_al / len(gold_all):6.1%} of target words; projected UPOS correct on aligned {n_ok / max(n_al, 1):6.1%}, "
              f"on all {n_ok / len(gold_all):6.1%} (unaligned = no label)")
        print(f"  {'':<20} unaligned by gold tag: {dict(unal.most_common(4))}; top confusions (projected, gold): {conf.most_common(3)}")

## Worked solution

Output of `experiments/ch17_exercises/run_ch17_exercises.py`:

```
UD 2.17 TueCL test files: tr 148, az 148, uz 148, ky 173 sentences; aligned through parallel_id

Turkish -> Azerbaijani: 148 parallel sentences, 912 target words; majority tag NOUN = 26.6%
  string+position      aligned  63.5% of target words; projected UPOS correct on aligned  95.9%, on all  60.9% (unaligned = no label)
                       unaligned by gold tag: {'NOUN': 118, 'VERB': 87, 'ADJ': 26, 'ADV': 25}; top confusions (projected, gold): [(('AUX', 'VERB'), 5), (('PROPN', 'NOUN'), 4), (('ADV', 'NOUN'), 2)]
  XLM-R mutual argmax  aligned  74.2% of target words; projected UPOS correct on aligned  93.4%, on all  69.3% (unaligned = no label)
                       unaligned by gold tag: {'NOUN': 79, 'VERB': 63, 'ADJ': 17, 'ADV': 17}; top confusions (projected, gold): [(('AUX', 'VERB'), 10), (('PROPN', 'NOUN'), 4), (('VERB', 'NOUN'), 3)]

Turkish -> Uzbek: 148 parallel sentences, 939 target words; majority tag NOUN = 22.8%
  string+position      aligned  42.0% of target words; projected UPOS correct on aligned  95.7%, on all  40.1% (unaligned = no label)
                       unaligned by gold tag: {'VERB': 172, 'NOUN': 146, 'ADJ': 49, 'AUX': 36}; top confusions (projected, gold): [(('AUX', 'VERB'), 6), (('VERB', 'NOUN'), 3), (('PROPN', 'NOUN'), 2)]
  XLM-R mutual argmax  aligned  48.6% of target words; projected UPOS correct on aligned  90.8%, on all  44.1% (unaligned = no label)
                       unaligned by gold tag: {'VERB': 154, 'NOUN': 114, 'ADJ': 44, 'PUNCT': 41}; top confusions (projected, gold): [(('PROPN', 'NOUN'), 3), (('PRON', 'NOUN'), 3), (('VERB', 'NOUN'), 3)]

Turkish -> Kyrgyz: 129 parallel sentences, 901 target words; majority tag NOUN = 21.1%
  string+position      aligned  39.2% of target words; projected UPOS correct on aligned  96.9%, on all  38.0% (unaligned = no label)
                       unaligned by gold tag: {'NOUN': 144, 'VERB': 136, 'AUX': 72, 'ADV': 48}; top confusions (projected, gold): [(('AUX', 'PART'), 3), (('NOUN', 'VERB'), 2), (('NOUN', 'PRON'), 1)]
  XLM-R mutual argmax  aligned  60.6% of target words; projected UPOS correct on aligned  88.5%, on all  53.6% (unaligned = no label)
                       unaligned by gold tag: {'VERB': 78, 'NOUN': 59, 'AUX': 52, 'PROPN': 40}; top confusions (projected, gold): [(('AUX', 'ADV'), 9), (('AUX', 'PART'), 8), (('AUX', 'VERB'), 8)]
```

**Report coverage and tag agreement together.** Depending on the target and aligner, projected UPOS matches gold on 88.5–96.9% of aligned words. Across all target words, counting unaligned words as unlabelled, the share is 38.0–69.3%. These denominators include punctuation.

These are projected-tag scores, not alignment precision: two incorrectly linked words may share a UPOS tag. The XLM-R method links more words but has lower tag agreement on linked words than the string method in these samples. Inspect individual links and construction differences before explaining why.

Only shared `parallel_id` items are evaluated: 148 Turkish–Azerbaijani and Turkish–Uzbek pairs, and 129 Turkish–Kyrgyz pairs. The string method converts the targets to CTS but leaves Turkish in its native spelling. The embedding method truncates at 256 tokens and gives missing words zero vectors, which mutual argmax does not explicitly exclude. Audit truncation before applying it to longer text.

No task model is trained. If these projected labels are later used for training, reserve different sentences for evaluation. The selected parallel examples do not represent unrestricted natural text.

## Your turn — extension

Project dependency heads as well as UPOS for the aligned words, count cycles and unattached words, and compare the projected trees with the gold Kyrgyz trees with an attachment score restricted to aligned words.